In [ ]:
# loading pdfs
from langchain_community.document_loaders import PyPDFLoader # better, faster for pure text
from langchain_community.document_loaders import UnstructuredFileLoader # better for also understanding figures, tables
import os, glob

# load paths
path_to_database = "path to textbook and paper pdfs"  # Change this to your path to the database

# create list with all pdf files
initial_pdf_path = glob.glob(os.path.join(path_to_database, 'textbook_pdf', '*.pdf'))
initial_pdf_path += glob.glob(os.path.join(path_to_database, 'paper_pdf', '**', '*.pdf'))
remove_dup = ("1-432.pdf", "433-680.pdf", "8-178.pdf", "776-857.pdf")
pdf_path = [f for f in initial_pdf_path if not f.endswith(remove_dup)]

print(len(initial_pdf_path))
print(len(pdf_path))

In [ ]:
# parse pdfs into langchain document objects
documents = []
for file_path in pdf_path:
    loader = PyPDFLoader(file_path)
    documents.extend(loader.load())

# check if properly loaded
if documents:
    print(documents[10].metadata)
    print(documents[10].page_content)

In [ ]:
# text splitting and chunking
from langchain.text_splitter import RecursiveCharacterTextSplitter # good for paragraph understanding

chunk = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200) # character chunk size and overlap
chunked = chunk.split_documents(documents)
# print(chunked)
# for c in chunked:
#     print(c.page_content)

In [ ]:
# --- Embedding model: build clean chunks, drop invalid ones, then encode -----
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import math, re, unicodedata as ud
from transformers import AutoTokenizer

# Load model (GPU fp16 ok)
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

# ---------- cleaners ----------
# control chars (keep \t \n \r), zero-widths, and UTF-16 surrogates
CTRL_CHARS = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
ZERO_WIDTH = re.compile(r"[\u200B-\u200D\uFEFF]")
SURROGATES = re.compile(r"[\ud800-\udfff]")

def sanitize_text(s: str) -> str:
    s = ud.normalize("NFKC", s)
    s = CTRL_CHARS.sub("", s)
    s = ZERO_WIDTH.sub("", s)
    s = SURROGATES.sub("", s)
    # keep only valid utf-8 bytes (drops any lingering invalids)
    s = s.encode("utf-8", "ignore").decode("utf-8", "ignore")
    return s.strip()

MIN_CHARS, MAX_CHARS = 20, 6000  # bounds to avoid pathological inputs

# ---------- collect raw text/meta from `chunked` ----------
raw_text, raw_metadata = [], []
for c in chunked:
    pc = getattr(c, "page_content", None)
    if pc is None and isinstance(c, dict):
        pc = c.get("page_content")

    if pc is None or isinstance(pc, (dict, list, tuple, set)):
        continue
    if isinstance(pc, (bytes, bytearray)):
        try:
            pc = pc.decode("utf-8", errors="ignore")
        except Exception:
            continue
    elif isinstance(pc, (int, float)):
        if isinstance(pc, float) and math.isnan(pc):
            continue
        pc = str(pc)
    elif not isinstance(pc, str):
        try:
            pc = str(pc)
        except Exception:
            continue

    pc = sanitize_text(pc)
    if not pc or not (MIN_CHARS <= len(pc) <= MAX_CHARS):
        continue

    raw_text.append(pc)
    md = getattr(c, "metadata", {}) if hasattr(c, "metadata") else (
        c.get("metadata", {}) if isinstance(c, dict) else {}
    )
    raw_metadata.append(md)

print(f"initial keep: {len(raw_text)} valid chunks, dropped {len(chunked) - len(raw_text)} invalid chunks")

# ---------- probe with tokenizer & drop anything that still breaks ----------
probe_tok = AutoTokenizer.from_pretrained("BAAI/bge-m3")

good_text, good_meta = [], []
dropped = 0
for t, md in zip(raw_text, raw_metadata):
    try:
        enc = probe_tok(t, truncation=True, max_length=512)
        if not enc or not enc.input_ids:
            dropped += 1
            continue
    except Exception:
        dropped += 1
        continue
    good_text.append(t)
    good_meta.append(md)

raw_text, raw_metadata = good_text, good_meta
print(f"after tokenizer filter: kept {len(raw_text)} chunks, dropped {dropped} more")

# ---------- encode embeddings ----------
texts = list(raw_text)  # plain list[str]
encode_vec = model.encode(
    texts,
    batch_size=32,
    max_length=8192,     # lower to 4096/2048 if you hit OOM or truncation issues
    return_dense=True,
    return_sparse=False
)
dense_vec = np.asarray(encode_vec["dense_vecs"], dtype=np.float32)

# normalize for cosine similarity
norms = np.linalg.norm(dense_vec, axis=1, keepdims=True)
norms[norms == 0] = 1.0
dense_vec = dense_vec / norms

print("Vectors:", dense_vec.shape, "mean norm:", float(np.linalg.norm(dense_vec, axis=1).mean()))
print(f"Encoded {len(dense_vec)} vectors with dim = {len(dense_vec[0])}")

In [ ]:
# --- Store precomputed embeddings safely into ChromaDB -----------------------
from langchain_community.vectorstores import Chroma
import math

# Initialize (no embedding_function since we already encoded)
vectordb = Chroma(
    collection_name="your database name",
    embedding_function=None,
    persist_directory="./chromadb",
)

# Generate unique IDs for each chunk
chunk_name = [f"chunk{i}" for i in range(len(raw_text))]

# Define a helper to add in safe batches (< 5000)
def add_in_batches(collection, embeddings, documents, metadatas, ids, batch_size=5000):
    n = len(ids)
    for i in range(0, n, batch_size):
        j = min(i + batch_size, n)
        collection.add(
            embeddings=embeddings[i:j],
            documents=documents[i:j],
            metadatas=metadatas[i:j],
            ids=ids[i:j],
        )
        print(f"Added batch {math.ceil(j/batch_size)} ({i}:{j})")

# Use the helper instead of a single large add()
add_in_batches(
    vectordb._collection,
    embeddings=dense_vec,
    documents=raw_text,
    metadatas=raw_metadata,
    ids=chunk_name,
    batch_size=5000,  # stay below internal limit (5461 in your case)
)

# Persist to disk
vectordb.persist()
print("✅ ChromaDB stored successfully at ./chromadb")